In [ ]:
import re
import math
import numpy as np
from collections import Counter
from tkinter import *
from tkinter import scrolledtext, messagebox
import requests
import threading


# 1. Chunking



## Main Equations

Number of Words:

L = |W|

Chunk Start Position:

start_i = i × (C - O)

Chunk End Position:

end_i = start_i + C

Chunk Content:

Chunk_i = W[start_i : end_i]

Next Chunk Position:

start_(i+1) = start_i + (C - O)

Number of Chunks:

K = ⌈(L - O) / (C - O)⌉


Where:

L = Total number of words
W = List of words in the document
C = Chunk size
O = Overlap size
K = Number of chunks
i = Chunk index


### Main Idea

The document is first split into individual words:

W = [w₁, w₂, w₃, ..., w_L]

The words are then divided into smaller chunks of size C.

Each consecutive chunk shares O words with the previous chunk.

Step Size:

Step Size = C - O

The overlap helps preserve context between neighboring chunks.

In short:

Next Start = Current Start + Chunk Size - Overlap

Chunking = Fixed-size segments + Context overlap

In [ ]:
def chunk_text(text: str, chunk_size: int = 40, overlap: int = 10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# 2. Embeddings


## TF-IDF Explanation

### Main Equations

Document Frequency:

DF(w) = |{d : w ∈ d}|

Inverse Document Frequency:

IDF(w) = log((N + 1) / (DF(w) + 1)) + 1

Term Frequency:

TF(w,d) = Count(w,d) / |d|

TF-IDF:

TFIDF(w,d) = TF(w,d) × IDF(w)

Final Vector:

v_d = [TFIDF(w₁,d), TFIDF(w₂,d), ..., TFIDF(w_|V|,d)]

### Main Idea

TF tells us how important a word is inside one Document.

IDF tells us how rare or unique the word is across all Documents.

TF-IDF combines both:

TF-IDF = Local Importance × Global Rarity

Therefore, words that occur frequently within a Document but are not common across other Documents generally receive higher TF-IDF values.


In [ ]:
ENGLISH_STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "in", "on", "at", "to", "for", "of", "with", "and", "or", "but",
    "this", "that", "these", "those", "it", "its", "as", "by", "from",
    "has", "have", "had", "do", "does", "did", "will", "would", "can",
    "could", "should", "what", "which", "who", "how",
}


def tokenize(text: str):
    text = text.lower()
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in ENGLISH_STOPWORDS]
    return tokens


class TfidfEmbedder:

    def __init__(self):
        self.vocab = {}
        self.idf = {}

    def fit(self, documents: list[str]):
        # 1. Building the vocab
        doc_tokens = [tokenize(doc) for doc in documents]
        all_words = sorted(set(word for tokens in doc_tokens for word in tokens))
        self.vocab = {word: i for i, word in enumerate(all_words)}

        # 2. IDF
        n_docs = len(documents)
        doc_freq = Counter()
        for tokens in doc_tokens:
            for word in set(tokens):
                doc_freq[word] += 1

        for word in self.vocab:
            self.idf[word] = math.log((n_docs + 1) / (doc_freq[word] + 1)) + 1

        return self

    def transform(self, text: str) -> np.ndarray:
        tokens = tokenize(text)
        vector = np.zeros(len(self.vocab))

        # TF
        term_counts = Counter(tokens)
        total_terms = len(tokens) if tokens else 1

        for word, count in term_counts.items():
            if word in self.vocab:
                tf = count / total_terms
                idf = self.idf[word]
                vector[self.vocab[word]] = tf * idf

        return vector



# 3. Cosine Similarity + Retrieval



## Main Equations

Vector Norm:

||a|| = √(Σᵢ aᵢ²)

Dot Product:

a · b = Σᵢ aᵢbᵢ

Cosine Similarity:

CosineSimilarity(a,b) = (a · b) / (||a|| × ||b||)

Chunk Score:

scoreᵢ = CosineSimilarity(query, chunkᵢ)

Top-k:

TopK = highest k similarity scores




In [ ]:
def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:

    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))


def retrieve(query: str, chunks: list[str], chunk_vectors: list[np.ndarray],
             embedder: TfidfEmbedder, top_k: int = 3):

    query_vector = embedder.transform(query)

    scores = []
    for i, chunk_vec in enumerate(chunk_vectors):
      score = cosine_similarity(query_vector, chunk_vec)
      scores.append((i, score))
    scores.sort(key=lambda x: x[1], reverse=True)

    top_results = []
    for i, score in scores[:top_k]:
       chunk_text = chunks[i]
       top_results.append((chunk_text, score))

    return top_results

# 4. Augmentation


In [ ]:
def build_prompt(query: str, retrieved_chunks: list[tuple[str, float]], lang: str = "ar") -> str:
    context = "\n\n".join(f"- {chunk}" for chunk, _ in retrieved_chunks)

    if lang == "en":
        prompt = f"""Use ONLY the following information to answer the question. If the information is not enough, say "Not enough information available".

Retrieved context:
{context}

Question: {query}

Answer:"""
    else:
        prompt = f"""استخدم المعلومات التالية فقط عشان تجاوب على السؤال. لو المعلومات مش كفاية، قول "المعلومات المتاحة غير كافية".

المعلومات المسترجعة:
{context}

السؤال: {query}

الإجابة:"""
    return prompt



# 5. Generation


In [ ]:
import os

# Load your Hugging Face token from an environment variable (never hardcode it!)
# Create a ".env" file locally (see .env.example) or set the env var before running:
#   Windows (PowerShell):  $env:HF_TOKEN="your_token_here"
#   macOS/Linux:            export HF_TOKEN="your_token_here"
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
HF_MODEL = "microsoft/Phi-3-mini-4k-instruct:featherless-ai"


def generate_answer(prompt: str) -> str:
    import requests

    if HF_TOKEN == "YOUR_HF_TOKEN_HERE":
        return "[محتاج تحط الـ Hugging Face token بتاعك في متغير البيئة HF_TOKEN عشان يشتغل]"

    api_url = "https://router.huggingface.co/v1/chat/completions"
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}

    try:
        response = requests.post(
            api_url,
            headers=headers,
            json={
                "model": HF_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 200,
                "temperature": 0.3,
            },
            timeout=30,
        )
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"].strip()

    except requests.exceptions.RequestException as e:
        return f"[حصل خطأ في الاتصال بـ Hugging Face: {e}]"


# 6. Full Pipeline



In [ ]:

def run_rag_pipeline(document: str, query: str, top_k: int = 2, lang: str = "ar"):
    print("\n" + "_" * 65)
    print("                    RAG PIPELINE")
    print("_"  * 65)

    # 1. Chunking
    print("\n[1/5] CHUNKING")
    print("      Splitting the document into smaller chunks")
    print("-" * 65)

    chunks = chunk_text(document, chunk_size=25, overlap=5)

    print(f"      Total chunks: {len(chunks)}")
    for i, c in enumerate(chunks):
        print(f"\n      Chunk {i}")
        print(f"      {c[:100]}...")

    # 2. Embedding
    print("\n\n[2/5] EMBEDDING")
    print("      Converting chunks into TF-IDF vectors")
    print("-" * 65)

    embedder = TfidfEmbedder().fit(chunks)
    chunk_vectors = [embedder.transform(c) for c in chunks]

    print(f"      Vocabulary size : {len(embedder.vocab)} words")
    print(f"      Vector size     : {chunk_vectors[0].shape}")

    # 3. Retrieval
    print("\n\n[3/5] RETRIEVAL")
    print("      Finding the most relevant chunks for the query")
    print("-" * 65)

    results = retrieve(query, chunks, chunk_vectors, embedder, top_k=top_k)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n      Result {i}  |  Similarity: {score:.3f}")
        print(f"      {chunk[:100]}...")

    # 4. Augmentation
    print("\n\n[4/5] AUGMENTATION")
    print("      Building the final prompt using retrieved context")
    print("-" * 65)

    prompt = build_prompt(query, results, lang=lang)

    print("\n      Final Prompt:")
    print("      " + "-" * 55)
    print(prompt)
    print("      " + "-" * 55)

    # 5. Generation
    print("\n\n[5/5] GENERATION")
    print("      Sending the prompt to the language model")
    print("-" * 65)

    answer = generate_answer(prompt)

    print("\n      Model Answer:")
    print("      " + "-" * 55)
    print(f"      {answer}")
    print("      " + "-" * 55)

    print("\n" + "_" * 65)
    print("                    PIPELINE COMPLETE")
    print("_" * 65)

    return answer


In [ ]:
if __name__ == "__main__":
    # English example
    sample_document = """
    Rewire AI is an adaptive learning platform designed to support students
    with diverse cognitive profiles. The project is built by a five-member
    team and uses Phi-3 as its core language model. The system includes a
    Cognitive Digital Twin module that builds a digital model of each
    student's learning style, an attention monitoring module built with
    MediaPipe, a text simplification module, a spaced repetition module for
    reviewing information, and a task management module. The backend is
    built with FastAPI and PostgreSQL, while the frontend uses React and
    Next.js. The project's branding is based on an axolotl mascot, with a
    mint, coral, and teal color palette.
    """

    user_query = "How many people are on the team and which model do they use?"

    run_rag_pipeline(sample_document, user_query, top_k=2, lang="en")


In [ ]:


# ====================== GUI ======================

# ---- Color palette ----
BG_MAIN     = "#eaf2fb"   # light blue background (window)
BG_ACCENT   = "#d3e6fb"   # deeper blue for frames
BG_PANEL    = "#f5f9fe"   # near-white panel background
BLUE_DARK   = "#1a4d8f"   # deep blue for text/labels
BLUE_MED    = "#2f6fbd"   # medium blue for accents/borders
BLUE_LIGHT  = "#5b9bd5"   # lighter accent
WHITE       = "#ffffff"

FONT_TITLE  = ("Segoe UI", 20, "bold")
FONT_LABEL  = ("Segoe UI", 10, "bold")
FONT_BODY   = ("Segoe UI", 10)
FONT_MONO   = ("Consolas", 10)

root = Tk()
root.title("Simple RAG - GUI")
root.geometry("900x750")
root.resizable(True, True)
root.configure(bg=BG_MAIN)

# ---- Title ----
title_label = Label(root, text="🔷 Simple RAG Pipeline", font=FONT_TITLE,
                     bg=BG_MAIN, fg=BLUE_DARK)
title_label.pack(pady=15)

# ---- Document input ----
doc_frame = LabelFrame(root, text="1. Paste your Document", padx=10, pady=5,
                        bg=BG_ACCENT, fg=BLUE_DARK, font=FONT_LABEL,
                        highlightbackground=BLUE_MED, highlightthickness=1)
doc_frame.pack(fill="both", expand=True, padx=15, pady=5)

doc_text = scrolledtext.ScrolledText(doc_frame, height=8, wrap=WORD, font=FONT_BODY,
                                      bg=WHITE, fg=BLUE_DARK, insertbackground=BLUE_DARK,
                                      relief="flat", highlightthickness=1,
                                      highlightbackground=BLUE_MED, highlightcolor=BLUE_DARK)
doc_text.pack(fill="both", expand=True)

# Default example document
default_doc = """Rewire AI is an adaptive learning platform designed to support students with diverse cognitive profiles. The project is built by a five-member team and uses Phi-3 as its core language model. The system includes a Cognitive Digital Twin module that builds a digital model of each student's learning style, an attention monitoring module built with MediaPipe, a text simplification module, a spaced repetition module for reviewing information, and a task management module. The backend is built with FastAPI and PostgreSQL, while the frontend uses React and Next.js. The project's branding is based on an axolotl mascot, with a mint, coral, and teal color palette."""
doc_text.insert(END, default_doc)

# ---- Query input ----
query_frame = LabelFrame(root, text="2. Ask a Question", padx=10, pady=5,
                          bg=BG_ACCENT, fg=BLUE_DARK, font=FONT_LABEL,
                          highlightbackground=BLUE_MED, highlightthickness=1)
query_frame.pack(fill="x", padx=15, pady=5)

query_entry = Entry(query_frame, font=("Segoe UI", 12), bg=WHITE, fg=BLUE_DARK,
                     insertbackground=BLUE_DARK, relief="flat",
                     highlightthickness=1, highlightbackground=BLUE_MED,
                     highlightcolor=BLUE_DARK)
query_entry.pack(fill="x", padx=5, pady=5, ipady=4)
query_entry.insert(0, "How many people are on the team and which model do they use?")

# ---- Settings ----
settings_frame = Frame(root, bg=BG_ACCENT, bd=1, relief="solid",
                        highlightbackground=BLUE_MED, highlightthickness=1)
settings_frame.pack(fill="x", padx=15, pady=8, ipady=6)

Label(settings_frame, text="⚙ Top-K:", bg=BG_ACCENT, fg=BLUE_DARK,
      font=FONT_LABEL).pack(side=LEFT, padx=(10, 2))

topk_var = StringVar(value="2")
topk_entry = Entry(settings_frame, textvariable=topk_var, width=5,
                    bg=WHITE, fg=BLUE_DARK, font=FONT_BODY,
                    relief="flat", highlightthickness=1,
                    highlightbackground=BLUE_MED, highlightcolor=BLUE_DARK,
                    justify="center")
topk_entry.pack(side=LEFT, padx=5, ipady=3)

Label(settings_frame, text="🌐 Language:", bg=BG_ACCENT, fg=BLUE_DARK,
      font=FONT_LABEL).pack(side=LEFT, padx=(20, 2))

lang_var = StringVar(value="en")
Radiobutton(settings_frame, text="English", variable=lang_var, value="en",
            bg=BG_ACCENT, fg=BLUE_DARK, selectcolor=WHITE,
            activebackground=BG_ACCENT, font=FONT_BODY,
            indicatoron=True).pack(side=LEFT, padx=5)

Radiobutton(settings_frame, text="Arabic", variable=lang_var, value="ar",
            bg=BG_ACCENT, fg=BLUE_DARK, selectcolor=WHITE,
            activebackground=BG_ACCENT, font=FONT_BODY,
            indicatoron=True).pack(side=LEFT, padx=5)

# ---- Run button ----
def run_pipeline():
    document = doc_text.get("1.0", END).strip()
    query = query_entry.get().strip()

    if not document or not query:
        messagebox.showwarning("Missing input", "Please paste a document and type a question.")
        return

    try:
        top_k = int(topk_var.get())
    except:
        top_k = 2

    # Clear previous results
    result_text.delete("1.0", END)
    result_text.insert(END, "Running RAG pipeline... please wait\n\n")
    root.update()

    def worker():
        try:
            # 1. Chunking
            chunks = chunk_text(document, chunk_size=25, overlap=5)
            result_text.insert(END, f"[1] CHUNKING → {len(chunks)} chunks created\n")
            for i, c in enumerate(chunks):
                result_text.insert(END, f"   Chunk {i}: {c[:80]}...\n")
            result_text.insert(END, "\n")

            # 2. Embedding
            embedder = TfidfEmbedder().fit(chunks)
            chunk_vectors = [embedder.transform(c) for c in chunks]
            result_text.insert(END, f"[2] EMBEDDING → Vocabulary size: {len(embedder.vocab)} words\n\n")

            # 3. Retrieval
            results = retrieve(query, chunks, chunk_vectors, embedder, top_k=top_k)
            result_text.insert(END, "[3] RETRIEVAL → Top results:\n")
            for i, (chunk, score) in enumerate(results, 1):
                result_text.insert(END, f"   Result {i} | Similarity: {score:.3f}\n")
                result_text.insert(END, f"   {chunk[:100]}...\n\n")

            # 4. Augmentation
            prompt = build_prompt(query, results, lang=lang_var.get())
            result_text.insert(END, "[4] AUGMENTATION → Prompt built\n\n")

            # 5. Generation
            result_text.insert(END, "[5] GENERATION → Calling model...\n")
            root.update()
            answer = generate_answer(prompt)

            result_text.insert(END, "\n" + "="*50 + "\n")
            result_text.insert(END, "FINAL ANSWER:\n")
            result_text.insert(END, answer + "\n")
            result_text.insert(END, "="*50 + "\n")

        except Exception as e:
            result_text.insert(END, f"\nError: {e}\n")

    # Run in a thread so the GUI doesn't freeze
    threading.Thread(target=worker, daemon=True).start()

run_btn = Button(root, text="▶  Run RAG Pipeline", font=("Segoe UI", 12, "bold"),
                 bg=BLUE_MED, fg="white", activebackground=BLUE_DARK,
                 activeforeground="white", relief="flat",
                 padx=20, pady=8, cursor="hand2", command=run_pipeline)
run_btn.pack(pady=10)

# ---- Results area ----
result_frame = LabelFrame(root, text="3. Results", padx=10, pady=5,
                           bg=BG_ACCENT, fg=BLUE_DARK, font=FONT_LABEL,
                           highlightbackground=BLUE_MED, highlightthickness=1)
result_frame.pack(fill="both", expand=True, padx=15, pady=5)

result_text = scrolledtext.ScrolledText(result_frame, height=12, wrap=WORD, font=FONT_MONO,
                                          bg=BG_PANEL, fg=BLUE_DARK,
                                          insertbackground=BLUE_DARK, relief="flat",
                                          highlightthickness=1, highlightbackground=BLUE_MED,
                                          highlightcolor=BLUE_DARK)
result_text.pack(fill="both", expand=True)

# ---- Clear button ----
def clear_all():
    doc_text.delete("1.0", END)
    query_entry.delete(0, END)
    result_text.delete("1.0", END)

clear_btn = Button(root, text="🗑 Clear", font=("Segoe UI", 10),
                    bg=BLUE_LIGHT, fg="white", activebackground=BLUE_MED,
                    activeforeground="white", relief="flat",
                    padx=15, pady=4, cursor="hand2", command=clear_all)
clear_btn.pack(pady=5)

# Start the GUI
root.mainloop()
